In [1]:
import os
import torch
import yaml
from DataLoaders.transforms import get_train_img_transform_1, get_train_img_transform_2, get_train_img_transform_3, get__val_test_img_transform, get_train_serie_transform

from DataLoaders.dataloaders import create_dataloader
from Losses import get_loss
from Metrics import get_metrics
from Model import get_model
from Training.evaluator import Evaluator
from Training.trainer import Trainer
from utils.logger import get_logger, TensorboardLogger
from utils.misc import get_device

##### Path to the config file #####
cfg_path = "/home/loai/Documents/code/RSMLExtraction/RSA_deep_working/Models/config.yml"
assert os.path.exists(cfg_path), f"Le fichier de config n'existe pas : {cfg_path}"
with open(cfg_path, "r") as f:
    config = yaml.safe_load(f)

##### Create dataloaders #####
train_loader, val_loader, test_loader, series_val_loader, series_test_loader = create_dataloader(
    base_directory=config["data"]["base_dir"],
    img_transforms=[
        get_train_img_transform_1(patch_size=512), 
        get_train_img_transform_2(patch_size=512), 
        get_train_img_transform_3(patch_size=512),
        get__val_test_img_transform(),
        get_train_serie_transform()
        ],
    default_batch_size=int(config["data"].get("batch_size", 32)),
    seed=42
)

##### Initialize model, loss, optimizer, metrics, logger, and evaluator #####
device = get_device(preferred=config["training"].get("device", "cuda"))

model = get_model(config["model"])
model = model.to(device)

criterion = get_loss(config["loss"])
try:
    criterion = criterion.to(device)
except:
    print("Warning: Loss function does not support moving to device. Continuing without moving it.")
    pass

optimizer_name = config["optimizer"]["name"]
lr = float(config["optimizer"]["learning_rate"])
weight_decay = float(config["optimizer"].get("weight_decay", 0))

match optimizer_name.lower():
    case "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    case "sgd":
        momentum = float(config["optimizer"].get("momentum", 0.9))
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    case "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    case _:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

# Factory design pattern to get metrics from str name in config.yaml
metrics_dict = get_metrics(config["metrics"])

# Logs
log_dir = config["training"]["log_dir"]
os.makedirs(log_dir, exist_ok=True)
log_model_path = log_dir + "/" + config["model"]["name"] + "_" + config["loss"]["name"]
os.makedirs(log_model_path, exist_ok=True)
if not os.path.exists(log_model_path + "/training.log"):
    open(log_model_path + "/training.log", "w").close()
logger = get_logger(log_model_path + "/training.log")

# TensorBoard logger
tb_log_dir = os.path.join(log_model_path, "tensorboard_logs")
os.makedirs(tb_log_dir, exist_ok=True)
tb_logger = TensorboardLogger(log_dir=tb_log_dir)

checkpoint_dir = config["training"]["checkpoint_dir"] + "/" + config["model"]["name"] + "_" + config["loss"]["name"]


# log all configurations in the logger
logger.info("Starting training with the following configurations:")
logger.info(f"Model: {config['model']}")
logger.info(f"Loss: {config['loss']}")
logger.info(f"Optimizer: {config['optimizer']}")
logger.info(f"Metrics: {config['metrics']}")
logger.info(f"Training configurations: {config['training']}")

2025-06-17 16:08:41.957808: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 16:08:41.966954: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750169321.978192  392974 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750169321.981835  392974 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750169321.990570  392974 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Total series: 29
Total images: 841
Number of Training series : 20
Number of Validation series : 5
Number of Testing series : 4
Number of Training images : 580
Number of Validation images : 145
Number of Testing images : 116

Number of transformed images: 841 (dataset 1), 841 (dataset 2), 841 (dataset 3)


In [2]:
serie_batch = next(iter(series_val_loader))
print("Serie batch shape:", serie_batch[0].shape, "Serie batch length:", len(serie_batch))
imgs, mask, time, mtg = serie_batch
print("Serie batch imgs shape:", imgs.shape, "mask shape:", mask.shape, time[0], mtg[0])

Serie batch shape: torch.Size([29, 1, 1166, 1348]) Serie batch length: 4
Serie batch imgs shape: torch.Size([29, 1, 1166, 1348]) mask shape: torch.Size([29, 1, 1166, 1348]) tensor(0) /home/loai/Images/DataTest/UC1_data/230629PN011/61_graph.rsml


In [3]:
from Metrics.mtg.area_below_intercep import AreaBetweenIntercepts
from rsml import rsml2mtg
from utils.intercept import intercept_curve_at_all_time

In [4]:
mtg_gt = rsml2mtg(mtg[0])
mtg_pred = rsml2mtg(mtg[1])

In [5]:
import numpy as np
def area_between_curves(x1, y1, x2, y2, num=1000) -> float:
        # bornes d'intersection des domaines
        x_min = max(min(x1), min(x2))
        x_max = min(max(x1), max(x2))
        x_common = np.linspace(x_min, x_max, num=num)
        y1i = np.interp(x_common, x1, y1)
        y2i = np.interp(x_common, x2, y2)
        return np.trapz(np.abs(y1i - y2i), x_common)

In [ ]:
plant_scale = 1
# On parcourt les mêmes sommets dans GT et PRED
verts_gt = list(mtg_gt.vertices(scale=plant_scale))
verts_pred = list(mtg_pred.vertices(scale=plant_scale))
print(verts_gt)
map_subtree_gt = {}
map_subtree_pred = {}
for v in verts_gt:
    map_subtree_gt[v] = mtg_gt.sub_mtg(v)
for v in verts_pred:
    map_subtree_pred[v] = mtg_pred.sub_mtg(v)

map_curve_gt = {}
for v in verts_gt:
    # courbe GT
    x_gt, y_gt = intercept_curve_at_all_time(map_subtree_gt[v], 0)
    map_curve_gt[v] = (x_gt, y_gt)

map_curve_pred = {}
for v in verts_pred:
    # courbe PRED
    x_pred, y_pred = intercept_curve_at_all_time(map_subtree_pred[v], 0)
    map_curve_pred[v] = (x_pred, y_pred)


Max scale in GT: 1
[1, 10, 20, 22, 27]


Computing intercepts: 100%|██████████| 2500/2500 [00:00<00:00, 6229.79it/s]


In [ ]:
#make a distance matrix between all curves in GT and PRED with area_between_curves
distance_matrix = np.zeros((len(verts_gt), len(verts_pred))) # line = GT, column = PRED
for i, v_gt in enumerate(verts_gt):
    x_gt, y_gt = map_curve_gt[v_gt]
    print(x_gt.shape, y_gt.shape)
    for j, v_pred in enumerate(verts_pred):
        x_pred, y_pred = map_curve_pred[v_pred]
        area = 0
        
        for t in range(y_gt.shape[0]): # 29 times
            area += area_between_curves(x_gt, y_gt[t, :], x_pred, y_pred[t, :])
        distance_matrix[i, j] = area
print("Distance matrix shape:", distance_matrix.shape)
print("Distance matrix:\n", distance_matrix)

(2500,) (29, 2500)
(2500,) (29, 2500)
(2500,) (29, 2500)
(2500,) (29, 2500)
(2500,) (29, 2500)
Distance matrix shape: (5, 5)
Distance matrix:
 [[ 0.         22.27627652 13.14400067 16.04840917 18.75160081]
 [22.27627652  0.         27.18803857 20.65244147 12.61636414]
 [13.14400067 27.18803857  0.          8.89118519 23.39719709]
 [16.04840917 20.65244147  8.89118519  0.         18.93234744]
 [18.75160081 12.61636414 23.39719709 18.93234744  0.        ]]


# version avec dtw

In [17]:
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

In [47]:
# Compute a DTW distance matrix between all curves in GT and PRED for each time step
num_timesteps = y_gt.shape[0]  # Should be 29
distance_matrix = np.zeros((len(verts_gt), len(verts_pred)))
for i, v_gt in enumerate(verts_gt):
    x_gt, y_gt = map_curve_gt[v_gt]
    print("GT curve shapes:", x_gt.shape, y_gt.shape)
    for j, v_pred in enumerate(verts_pred):
        x_pred, y_pred = map_curve_pred[v_pred]
        dtw = 0
        for t in range(num_timesteps):
            y_gt_t = y_gt[t, :]
            y_pred_t = y_pred[t, :]
            dtw += fastdtw(y_gt_t, y_pred_t)[0] # 0 for distance, 1 for path
        distance_matrix[i, j] = dtw
print("Distance matrix shape:", distance_matrix.shape)
print("Distance matrix:\n", distance_matrix)

GT curve shapes: (2500,) (29, 2500)
GT curve shapes: (2500,) (29, 2500)
GT curve shapes: (2500,) (29, 2500)
GT curve shapes: (2500,) (29, 2500)
GT curve shapes: (2500,) (29, 2500)
Distance matrix shape: (5, 5)
Distance matrix:
 [[    0.  4076.  4366.  1716.  2338.]
 [ 4146.     0. 10898.  6165.  3871.]
 [ 4162. 10623.     0.   553. 10686.]
 [ 1712.  6012.  1121.     0.  5683.]
 [ 2422.  3841. 10933.  5660.     0.]]
